# Inventory Optimization with GRPO + OpenEnv

Fine-tune **Qwen2.5-3B-Instruct** with **GRPO (Group Relative Policy Optimization)** to make inventory reorder-point decisions in a stochastic supply-chain environment.

- **Environment**: OpenEnv-compatible Inventory Simulation deployed on HF Spaces
- **Training**: Unsloth + HuggingFace TRL GRPOTrainer
- **Reward**: Analytical P&L + fill-rate composite

In [ ]:
# Install dependencies
!pip install -q "openenv-core[core]>=0.2.1" torch transformers trl peft accelerate datasets bitsandbytes

In [ ]:
import json, re, time
import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

## 1. Connect to OpenEnv Environment on HF Spaces

The inventory environment is deployed as an OpenEnv server. We use the
HTTP API to `reset()` and `step()` through episodes.

In [ ]:
import httpx

ENV_URL = "https://ademarteau-rl-inventory-simulations.hf.space"

def env_reset(env_type: int = 0) -> dict:
    r = httpx.post(f"{ENV_URL}/reset", json={"env_type": env_type}, timeout=30)
    r.raise_for_status()
    return r.json()["observation"]

def env_step(reorder_point: float, reasoning: str = "") -> dict:
    r = httpx.post(f"{ENV_URL}/step", json={"action": {"reorder_point": reorder_point, "reasoning": reasoning}}, timeout=30)
    r.raise_for_status()
    return r.json()

# Quick smoke test
obs = env_reset(env_type=0)
print(f"Day {obs['day']} | Inventory: {obs['current_inventory']:.0f} | Days remaining: {obs['days_remaining']}")

## 2. Prompt & Reward Setup

In [ ]:
SYSTEM_PROMPT = """You are an inventory optimization agent. Each step you receive the current state \
of a supply-chain simulation and must decide a reorder_point — the inventory level \
that triggers a replenishment order.

Reply with ONLY a JSON object:
{"reorder_point": <number>, "reasoning": "<brief explanation>"}"""


def format_user_msg(obs: dict) -> str:
    return json.dumps({
        "day": obs["day"], "days_remaining": obs["days_remaining"],
        "current_inventory": round(obs["current_inventory"], 1),
        "demand_mean_30d": round(obs["demand_mean_30d"], 1),
        "demand_std_30d": round(obs.get("demand_std_30d", 0), 1),
        "fill_rate_so_far": round(obs["fill_rate_so_far"], 4),
        "recent_stockouts": obs["recent_stockouts"],
        "pending_orders": obs.get("pending_orders", []),
    }, separators=(",", ":"))


def parse_rop(text: str):
    try:
        return float(json.loads(re.sub(r"```(?:json)?|```", "", text).strip())["reorder_point"])
    except Exception:
        m = re.search(r'"reorder_point"\s*:\s*([0-9]+\.?[0-9]*)', text)
        return float(m.group(1)) if m else None

In [ ]:
# Analytical reward: 30-day lookahead simulation
SP, UC, FC, HR, WR, LT = 25.0, 10.0, 150.0, 0.005, 0.00143, 3

def analytical_reward(obs: dict, rop: float) -> float:
    inv = obs["current_inventory"]
    md = obs["demand_mean_30d"]
    sd = obs.get("demand_std_30d", 0.0)
    horizon = min(30, obs.get("days_remaining", 365))
    if horizon <= 0 or md <= 0:
        return 0.0
    pending = [(p["arrival_day"], p["quantity"]) for p in obs.get("pending_orders", [])]
    profit = sold = demanded = 0.0
    for t in range(horizon):
        d = obs["day"] + t
        inv += sum(q for a, q in pending if a == d)
        pending = [(a, q) for a, q in pending if a > d]
        inv = max(0, inv - inv * WR)
        dem = md + (sd * 0.3 if t % 5 == 0 else 0)
        s = min(dem, inv); lost = max(0, dem - inv); inv = max(0, inv - dem)
        sold += s; demanded += dem
        pipe = sum(q for a, q in pending)
        oq = max(0, rop - (inv + pipe) + md * LT) if inv + pipe <= rop else 0
        if oq > 0: pending.append((d + LT, oq))
        profit += s*SP - inv*UC*HR - lost*(SP-UC) - (FC if oq>0 else 0) - oq*UC - inv*WR*UC
    base = md * (SP - UC) * horizon
    pnl = profit / base if base > 0 else 0
    fr = sold / demanded if demanded > 0 else 0
    fill = 1.0 if fr >= 0.95 else -(0.95 - fr) / 0.95
    return float(np.clip(0.6 * pnl + 0.4 * fill, -2, 2))


def reward_fn(completions, obs_json=None, **kw):
    rewards = []
    for i, c in enumerate(completions):
        rop = parse_rop(c)
        if rop is None:
            rewards.append(-1.0); continue
        if obs_json:
            raw = obs_json[i]
            obs = json.loads(raw[0] if isinstance(raw, list) else raw)
            rewards.append(analytical_reward(obs, rop))
        else:
            rewards.append(0.0)
    return rewards

print("Reward function ready.")

## 3. Collect Rollout Data via OpenEnv

In [ ]:
def collect_episode(tokenizer, env_type=0):
    """Run one heuristic episode and collect (prompt, obs_json) pairs."""
    obs = env_reset(env_type)
    rows = []
    done = False
    while not done:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": format_user_msg(obs)}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        rop = obs["demand_mean_30d"] * 3 + obs.get("demand_std_30d", 0) * 1.65
        result = env_step(rop)
        rows.append({"prompt": prompt, "obs_json": json.dumps(obs)})
        obs = result["observation"]
        done = result["done"]
    print(f"  Episode done: {len(rows)} steps, final fill rate: {obs['fill_rate_so_far']:.3f}")
    return rows

## 4. Load Model with HuggingFace Transformers + PEFT LoRA

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

## 5. GRPO Training Loop

In [ ]:
N_ITERATIONS = 3
losses = []

for it in range(N_ITERATIONS):
    print(f"\n{'='*50}")
    print(f"ITERATION {it+1}/{N_ITERATIONS}")
    print(f"{'='*50}")

    print("Collecting episode via OpenEnv HF Space...")
    rows = collect_episode(tokenizer, env_type=0)
    dataset = Dataset.from_list(rows)

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_fn,
        args=GRPOConfig(
            output_dir=f"./grpo_inventory/iter_{it}",
            num_train_epochs=1,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_generations=4,
            max_completion_length=128,
            learning_rate=5e-6,
            bf16=True,
            logging_steps=5,
            save_strategy="no",
            report_to="none",
        ),
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    result = trainer.train()
    loss = result.training_loss if hasattr(result, "training_loss") else 0
    losses.append(loss)
    print(f"  Loss: {loss:.4f}")

    model.save_pretrained(f"./grpo_inventory/iter_{it}")
    tokenizer.save_pretrained(f"./grpo_inventory/iter_{it}")

print(f"\nTraining complete. Losses: {losses}")

## 6. Plot Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(losses)+1), losses, 'o-', linewidth=2, markersize=8)
plt.xlabel("Iteration")
plt.ylabel("Training Loss")
plt.title("GRPO Training Loss — Inventory Optimization")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluate Fine-Tuned Model vs Baseline

In [ ]:
import torch

def run_eval_episode(model, tokenizer, env_type=0, use_model=True):
    """Run one full episode. If use_model=False, uses heuristic baseline."""
    obs = env_reset(env_type)
    done = False
    rewards = []
    while not done:
        if use_model:
            msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": format_user_msg(obs)}]
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=128, do_sample=True,
                                     temperature=0.7, pad_token_id=tokenizer.eos_token_id)
            text = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            rop = parse_rop(text)
            if rop is None:
                rop = obs["demand_mean_30d"] * 3 + obs.get("demand_std_30d", 0) * 1.65
        else:
            rop = obs["demand_mean_30d"] * 3 + obs.get("demand_std_30d", 0) * 1.65

        result = env_step(rop)
        rewards.append(result.get("reward", 0))
        obs = result["observation"]
        done = result["done"]
    return {"fill_rate": obs["fill_rate_so_far"], "total_reward": sum(rewards),
            "stockouts": obs["recent_stockouts"]}

print("Evaluating baseline (heuristic)...")
baseline = run_eval_episode(model, tokenizer, use_model=False)
print(f"  Baseline — Fill rate: {baseline['fill_rate']:.4f} | Reward: {baseline['total_reward']:.1f} | Stockouts: {baseline['stockouts']}")

print("\nEvaluating fine-tuned model...")
finetuned = run_eval_episode(model, tokenizer, use_model=True)
print(f"  Model   — Fill rate: {finetuned['fill_rate']:.4f} | Reward: {finetuned['total_reward']:.1f} | Stockouts: {finetuned['stockouts']}")

print(f"\nImprovement: Fill rate {baseline['fill_rate']:.4f} → {finetuned['fill_rate']:.4f}")
print(f"             Reward   {baseline['total_reward']:.1f} → {finetuned['total_reward']:.1f}")